## Build Goal Refinement Graph

This notebook builds a goal refinement graph from the goals sampled from agglomerative clustering.

In [1]:
from openai import OpenAI

client = OpenAI()

def prompt_model(prompt):
    response = client.chat.completions.create(
      model="gpt-5.2-2025-12-11",
      #model="gpt-4o-2024-08-06",
      messages=[
        {
          "role": "system",
          "content": "You are a business analyst building a goal model of stakeholder goals for a software application. Goal models are directed, acyclic graphs in which edges trace from high-level goals to low-level goals through refinement relationships. High-level goals describe what states stakeholders want to achieve, maintain or avoid in the system. Low-level goals describe *how* the system will satsify high-level goals, tend to be more specific and describe how the system will operate. High-level goals describe *why* the system aims to satisfy low-level goals, tend to be more generic and describe what the stakeholder aims to accomplish independent of a specific software application. "
        },
        {
          "role": "user",
          "content": prompt
        }
      ]
    )
    return response.choices[0].message.content

In [2]:
import json
import os

data_path = 'data1_gpt52'

cluster_27 = json.load(open(os.path.join(

In [3]:
for i, g in enumerate(sampled['5']):
    print('%s: %s' % (i, g))

0: Read books and articles digitally
1: Read on a tablet instead of a computer
2: Avoid a cumbersome online reading experience
3: Maintain focus while reading
4: Access and read public web content online
5: Provide a dedicated method for online reading in a school setting
6: Read news online from official sources
7: Store books and course materials in a personal online library.
8: Store library content across multiple storage providers.
9: Access previously stored course materials after a course or semester ends.
10: Use free-trial access to read news content without paying for an ongoing subscription
11: Access reading materials without needing to download and upload them
12: Download course-related PDF documents from web pages and course portals
13: Open and read PDF and eBook documents in a single interface
14: Manage PDF documents
15: Export PDF documents to another document reader
16: Open documents, including PDF files, on a tablet
17: Import course content to a tablet with fewer

In [4]:
# The program describes a goal graph, in which each goal describes a state in the system to be achieved, maintained or avoided. The functions define refinement relationships between two goals X and Y. When goal X is refined by goal Y, then goal Y must be satisfied in order to satisfy the goal X. Refinement relationships are asymmetric. When a goal X is refined by a goal Y, then the goal Y explains how the goal X is satisfied. The goal Y could be one of several steps that must be performed to satisfy goal X, or it could be one of many cases that must be handled to satisfy goal X. Refinement relationships are transitive, thus if a goal Z satisfies goal Y and goal Y satisfies goal X, then it is also true that Z satisfies X. 

prompt_base = """Read the following Python code that describes an initial goal model and complete the code using the following function calls. Before completing the code, perform the following steps: 1) read all of the goals and describe what the software application is and does; 2) for each goal, describe what the goal means in the context of the application description, including what is and is not intended by the goal description; and 3) complete the code by considering which goals are refined by other goals. Include your justification for each function call in comments. When responding, include the python code between the start ```python and end ``` tags.
1. X.is_refined_by.append(Y) - when the goal X is satisfied by the goal Y and has the refinement goal Y
2. Y.is_refinement_of(X) - when the goal Y satisfies the goal X and is a refinement of goal X
%s
"""

prompt_text = """Read the following Python code that describes an initial goal model and complete the code using the following function calls. Before completing the code, perform the following steps: 1) read all of the goals and describe what the software application is and does; 2) for each goal, describe what the goal means in the context of the application description, including what is and is not intended by the goal description; and 3) complete the code by adding your implied goals and then considering which goals are refined by other goals. Include your justification for each function call in comments. When responding, include the python code between the start ```python and end ``` tags.
1. X.is_why_we_satisfy_the_goal.append(Y) - when the goal X is satisfied by the goal Y and has the refinement goal Y
2. Y.is_how_we_satisfy_the_goal(X) - when the goal Y satisfies the goal X and is a refinement of goal X
%s
"""

prompt_w_implied = """Read the following Python code that describes an initial goal model and complete the code using the following function calls. Before completing the code, perform the following steps: 1) read all of the goals and describe what the software application is and does; 2) for each goal, describe what the goal means in the context of the application description, including what is and is not intended by the goal description; 3) identify up to two or three *implied* goals that were not included in the original goal list and that add missing context to the original goals; 4) extend the code by adding your implied goals; and 5) complete the code by reviewing all of the goals and deciding which goals are refined by other goals. Implied goals include high-level goals that group related refinements together, and explain what actions the low-level goals seek to achieve. For each implied goal that you create, add the goal to the list of implied goals. Include your justification for each function call in comments. Do not write code to print the goals. When responding, include the python code between the start ```python and end ``` tags.
1. X.is_refined_by.append(Y) - when the goal X is satisfied by the goal Y and has the refinement goal Y
2. Y.is_refinement_of(X) - when the goal Y satisfies the goal X and is a refinement of goal X
3. implied_goals.append(X) - to collect the implied goals that you created
%s
"""

def sample_code(prefix_code, verbose=False):
    p = prompt_w_implied % '\n'.join(prefix_code)
    if verbose:
        print(p)
    r = prompt_model(p)
    if verbose:
        print(r)

    return p, r

In [5]:
# create the class definition code lines
def create_class_def():
    class_def_base = [
        'class Goal:', 
        '    def __init__(self, text):', 
        '        self.text = text', 
        '        self.is_refined_by = []',
        '    def is_refinement_of(self, goal):',
        '        goal.is_refined_by.append(self)'
    ]
    class_def_text = [
        'class Goal:', 
        '    def __init__(self, text):', 
        '        self.text = text', 
        '        self.is_why_we_satisfy_the_goal = []',
        '    def is_how_we_satisfy_the_goal(self, goal):',
        '        goal.is_why_we_satisfy_the_goal.append(self)'
    ]
    class_def_implied = [
        'class Goal:', 
        '    def __init__(self, text):', 
        '        self.text = text', 
        '        self.is_refined_by = []',
        '    def is_refinement_of(self, goal):',
        '        goal.is_refined_by.append(self)',
        'implied_goals = []'
    ]
    return class_def_implied

# create the goal instances
def create_prefix_code(g_list, var_map):
    prefix_code = []
    for g in g_list:
        prefix_code.append('%s = Goal("%s")' % (var_map[g], g))
    return prefix_code

# extract code from model response
def match_code(text):
    match = []
    i = text.find('```python\n')
    while i >= 0:
        j = text.find('```', i + 1)
        match.extend(text[i+10:j].split('\n'))
        i = text.find('```python\n', j + 3)
    return match

In [6]:
def collect_responses(goals, verbose=False):
    # create initial class definition
    code = create_class_def()

    # create variable name map
    var_map = {g: 'g%s' % i for i, g in enumerate(goals)}

    # create goal definitions
    prefix_code = create_prefix_code(goals, var_map)

    response = {'pass': [], 'fail': []}
    passed = False
    failed = 0
    while not passed and failed < 3:
        try:
            p, r = sample_code(code + prefix_code)
            if verbose:
                print(p)
                print(r)
                print()

            # parse suffix code from response
            suffix_code = match_code(r)
            if len(suffix_code) == 0:
                raise Exception('Response did not a contain python match.')
                
            # include prefix, which may be omitted in response
            program = code + prefix_code + suffix_code
        
            # create symbol table from generated code
            local = {}
            exec('\n'.join(program), {}, local)
            passed = True
                
            # save responses for post-processing
            response['pass'] = [p, r, program]

            # append code for follow-on prompt
            code += prefix_code + suffix_code

            print(' .', end='')
        
        except Exception as e:
            if verbose:
                print('Error: %s' % e)
                print(p)
                print(r)
                print()
            response['fail'].append([p, r, str(e)])
            failed += 1
            print(' !', end='')
            
    if not passed:
        print(' F', end='')
                
    return response

In [7]:
import os

results = {}
for i, goals in sampled.items():
    if os.path.exists('%s/cache/refinements-%s-implied.json' % (data_path, i)):
        r = json.load(open('%s/cache/refinements-%s-implied.json' % (data_path, i), 'r'))
        if len(r) > len(results):
            results = r
print(len(results))
for i, r in results.items():
    print('%s: %i' % (i, len(r)))

0


In [8]:
for i, goals in sampled.items():
    if i in results:
        continue
        
    results[i] = []
    print('Building graph goals %s (%i goals)' % (i, len(goals)), end='')
    for j in range(10):
        result = collect_responses(goals, verbose=False)
        results[i].append(result)

    json.dump(results, open('%s/cache/refinements-%s-implied.json' % (data_path, i), 'w+')) 
    print(' .done!')

Building graph goals 0 (37 goals) . . . . . . . . . . .done!
Building graph goals 1 (47 goals) . . . . . . . . . . .done!
Building graph goals 2 (42 goals) . . . . . . . . . . .done!
Building graph goals 3 (79 goals) . . . . . . . . . . .done!
Building graph goals 4 (56 goals) . . . . . . . . . . .done!
Building graph goals 5 (31 goals) . . . . . . . . . . .done!
Building graph goals 6 (78 goals) . . . . . . . . . . .done!
Building graph goals 7 (37 goals) . . . . . . . . . . .done!
Building graph goals 8 (78 goals) . . . . . . . . . . .done!
Building graph goals 9 (51 goals) . . . . . . . . . . .done!
Building graph goals 10 (42 goals) . . . . . . . . . . .done!
Building graph goals 11 (95 goals) . . . . . . . . . . .done!
Building graph goals 12 (30 goals)

KeyboardInterrupt: 

In [13]:
sampled.keys()

dict_keys(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33'])

In [17]:
ix = 12
while ix < len(sampled):
    i = str(ix)
    goals = sampled[i]
    if i in results:
        continue

    results[i] = []
    print('Building graph goals %s (%i goals)' % (i, len(goals)), end='')
    for j in range(10):
        result = collect_responses(goals, verbose=False)
        results[i].append(result)

    json.dump(results, open('%s/cache/refinements-%s-implied.json' % (data_path, i), 'w+')) 
    print(' .done!')
    ix += 1

Building graph goals 12 (30 goals) . . . . . . . . . . .done!
Building graph goals 13 (63 goals) . . . . . . . . . . .done!
Building graph goals 14 (69 goals) . . . . . . . . . . .done!
Building graph goals 15 (29 goals) . . . . . . . . . . .done!
Building graph goals 16 (63 goals) . . . . . . . . . . .done!
Building graph goals 17 (37 goals) . . . . . . . . . . .done!
Building graph goals 18 (30 goals) . . . . . . . . . . .done!
Building graph goals 19 (84 goals) . . . ! . . . . . . . .done!
Building graph goals 20 (58 goals) . . . . . . . . . . .done!
Building graph goals 21 (25 goals) . . . . . . . . . . .done!
Building graph goals 22 (61 goals) . . . . . . . . . . .done!
Building graph goals 23 (23 goals) . . . . . . . . . . .done!
Building graph goals 24 (18 goals) . . . . . . . . . . .done!
Building graph goals 25 (59 goals) . . . . . . . . . . .done!
Building graph goals 26 (83 goals) . . . . . . . . . . .done!
Building graph goals 27 (154 goals) . . . . . . . . . . .done!
Build

In [18]:
print(len(results))

34


In [19]:
results.keys()

dict_keys(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33'])

In [20]:
import json
json.dump(results, open('%s/models.json' % data_path, 'w'))

In [21]:
print(results['4'][2]['pass'][1])

### 1) What the software application is and does
The goals describe an **email and messaging application with an AI assistant layer**. It supports accessing email (via browser), reliably receiving mail, organizing and triaging an inbox, searching/filtering, summarizing and highlighting important content, and composing/sending emails (including scheduling, bulk send, groups/BCC, forwarding). It also includes productivity features around follow-up and deadlines, and collaboration features using shared categories. Overall, it helps users **manage high volumes of email/messages efficiently and safely**, with emphasis on job/school/work communications and assistant-supported reading/writing.

---

### 2) Goal-by-goal meaning in context (what it means / what it does NOT mean)

- **g0 Help someone gain a deeper understanding of how users want to use a product**  
  *Means:* product/UX research goal: capture and analyze user needs for this email/messaging app.  
  *Not:* a runtime end-user fea